# VTuber Builder


In [ ]:
import pathlib, shutil, subprocess, sys, runpy

REPO_URL = "https://github.com/jujumelona/Virtual-pipeline.git"
REPO_DIR = pathlib.Path("/content/Virtual-pipeline")

def run(cmd, timeout):
    proc = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    if proc.returncode != 0:
        raise RuntimeError(
            f"Command failed: {' '.join(cmd)}\n"
            + (proc.stderr or proc.stdout or "")[-5000:]
        )
    return proc

# Always load the current public main branch before starting the UI.
if (REPO_DIR / ".git").is_dir():
    run(
        ["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL],
        60,
    )
    run(
        ["git", "-C", str(REPO_DIR), "fetch", "--prune", "origin", "main"],
        180,
    )
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run(
        [
            "git",
            "clone",
            "--branch",
            "main",
            "--single-branch",
            REPO_URL,
            str(REPO_DIR),
        ],
        300,
    )

run(
    ["git", "-C", str(REPO_DIR), "checkout", "-B", "main", "origin/main"],
    60,
)
run(
    ["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"],
    60,
)

# Colab regression in later Gradio 6.x releases: keep the known-good version.
run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--disable-pip-version-check",
        "gradio==6.3.0",
    ],
    600,
)

# UI and pipeline launcher are always taken from the freshly synchronized main.
runpy.run_path(
    str(REPO_DIR / "tools" / "colab_app.py"),
    run_name="__main__",
)
